# OMBRIA Confirmatory Smoke Gate

This two-epoch, one-seed run validates the complete execution path. Its scores are not scientific evidence.

Before **Run All**, enable a Kaggle GPU and Internet access. P100 is supported through an automatic CUDA 12.6 compatibility check.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/NewRudy/geoai-ombria-robustness.git"
TAG = "v0.1.5-confirmatory"
working = Path("/kaggle/working")
working.mkdir(parents=True, exist_ok=True)
os.chdir(working)
project = working / "geoai-ombria-robustness"
if project.exists():
    shutil.rmtree(project)
subprocess.run(["git", "clone", "--depth", "1", "--branch", TAG, REPO_URL, str(project)], check=True)
os.chdir(project)
print("locked source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24", "pillow>=10.0", "matplotlib>=3.7"], check=True)
subprocess.run([sys.executable, "scripts/ensure_cuda_compat.py"], check=True)
subprocess.run([sys.executable, "scripts/check_cuda_runtime.py"], check=True)


In [ ]:
subprocess.run([
    sys.executable, "-m", "py_compile",
    "scripts/train_ombria_unet.py",
    "scripts/evaluate_ombria_2021_events.py",
    "scripts/summarize_confirmatory_events.py",
    "scripts/export_confirmatory_event_panels.py",
    "scripts/ensure_cuda_compat.py",
    "scripts/check_cuda_runtime.py",
    "scripts/write_experiment_manifest.py",
    "scripts/package_confirmatory_artifacts.py",
], check=True)
subprocess.run(["bash", "-n", "scripts/run_confirmatory_event_matrix.sh"], check=True)
print("compile and shell gates: pass")


In [ ]:
env = os.environ.copy()
env.update({"MODE": "smoke", "EPOCHS": "2", "BATCH_SIZE": "8", "BASE_CHANNELS": "16", "PYTHON": sys.executable})
subprocess.run(["bash", "scripts/run_confirmatory_event_matrix.sh"], check=True, env=env)


In [ ]:
import hashlib
import json
from zipfile import ZipFile
from IPython.display import FileLink, display

artifact = project / "results/ombria_2021_confirmatory_artifacts.zip"
assert artifact.exists() and artifact.stat().st_size > 0, artifact
with ZipFile(artifact) as archive:
    assert archive.testzip() is None
    names = archive.namelist()
    artifact_manifest_name = next(name for name in names if name.endswith("artifact_manifest.json"))
    artifact_manifest = json.loads(archive.read(artifact_manifest_name))
    for record in artifact_manifest["files"]:
        assert record["path"] in names, record["path"]
        assert hashlib.sha256(archive.read(record["path"])).hexdigest() == record["sha256"]
    checkpoint_manifest_name = next(name for name in names if name.endswith("checkpoint_manifest.json"))
    checkpoint_manifest = json.loads(archive.read(checkpoint_manifest_name))
    assert len(checkpoint_manifest["checkpoints"]) == 5
assert any(name.endswith("tables/event_heldout_summary.csv") for name in names)
assert any(name.endswith("per_chip_metrics.csv") for name in names)
assert any(name.endswith("figures/panel_manifest.json") for name in names)
assert any(name.endswith("runs/multimodal_none_seed7/metrics.csv") for name in names)
assert any(name.endswith("runtime_manifest.json") for name in names)
assert any(name.endswith("experiment_manifest.json") for name in names)
assert any(name.endswith("artifact_manifest.json") for name in names)
assert any(name.endswith("checkpoint_manifest.json") for name in names)
assert any(name.endswith("environment_freeze.txt") for name in names)
assert any(name.endswith("run.log") for name in names)
print(f"artifact integrity and SHA-256 manifest: pass ({len(names)} files, {artifact.stat().st_size} bytes)")
display(FileLink(str(artifact)))
